# Lesson 17 — Exercises

Run `setup.sql` first, then work through `lesson.ipynb`. Solutions are in `exercises-solutions.ipynb`.

Several exercises need **two sessions** — in ADS, open a second query editor (or connection) to act as Session B (outside ADS: `.\scripts\connect.ps1`). Remember to COMMIT/ROLLBACK any open transaction when you finish a blocking demo.

In [ ]:
USE AdventureWorks2022;
GO
-- XML data type methods (used by the Exercise 4 deadlock-graph query) require QUOTED_IDENTIFIER ON.
SET QUOTED_IDENTIFIER ON;
GO

## Exercise 1

Create a blocking situation.
- **Session A:** `BEGIN TRAN; UPDATE lesson17.Account SET Balance = Balance - 50 WHERE AccountID = 2;` (leave open)
- **Session B:** `SELECT * FROM lesson17.Account WHERE AccountID = 2;` (this blocks)

Then, from a third session (or Session A in a new batch), write a query that reports the waiting session, who is blocking it, and the wait_type.

In [ ]:
SELECT r.session_id AS WaitingSession, r.blocking_session_id AS BlockedBy,
       r.wait_type, r.wait_resource, t.text AS WaitingSQL
FROM sys.dm_exec_requests AS r
CROSS APPLY sys.dm_exec_sql_text(r.sql_handle) AS t
WHERE r.blocking_session_id <> 0;

**Why:** `sys.dm_exec_requests` exposes blocking_session_id for waiting requests. The waiting session is Session B; BlockedBy is Session A's SPID; wait_type is typically LCK_M_S (it wants a Shared lock that Session A's Exclusive lock is holding off).

## Exercise 2

While the block from Exercise 1 is active, list the locks held by Session A, showing resource_type and request_mode. Identify the exclusive (X) lock.

In [ ]:
SELECT request_session_id, resource_type, request_mode, request_status,
       OBJECT_NAME(p.object_id) AS ObjectName
FROM sys.dm_tran_locks AS l
LEFT JOIN sys.partitions AS p ON p.hobt_id = l.resource_associated_entity_id
WHERE resource_database_id = DB_ID()
ORDER BY request_session_id, resource_type;

**Why:** `sys.dm_tran_locks`; the X lock on the KEY/RID is the one causing the block. Look for request_mode = 'X' on a KEY (or RID) resource held by Session A. Remember to COMMIT/ROLLBACK Session A to clear the block.

## Exercise 3

Run a large UPDATE on `lesson17.BigOrders` inside a transaction and show, from another session, that lock escalation produced an OBJECT-level lock. ROLLBACK afterwards.

In [ ]:
BEGIN TRANSACTION;
UPDATE lesson17.BigOrders SET Status = Status WHERE SalesOrderID > 0;
-- In another session, for this SPID:
--   SELECT resource_type, request_mode FROM sys.dm_tran_locks WHERE request_session_id = <SPID>;
--   Expect an 'OBJECT' resource_type with request_mode 'X' (escalated from many KEY locks).
ROLLBACK TRANSACTION;

**Why:** an UPDATE touching every row escalates to a single OBJECT (table) X lock.

## Exercise 4

Produce a deadlock using two sessions and the AccountID 1 / 2 ordering from the examples. Then query the system_health ring buffer for the deadlock graph.

In [ ]:
SELECT XEvent.value('(@timestamp)[1]', 'datetime2') AS DeadlockTime,
       XEvent.query('.')                            AS DeadlockGraphXml
FROM (
    SELECT CAST(target_data AS XML) AS TargetData
    FROM sys.dm_xe_session_targets st
    JOIN sys.dm_xe_sessions s ON s.address = st.event_session_address
    WHERE s.name = 'system_health' AND st.target_name = 'ring_buffer'
) AS Data
CROSS APPLY TargetData.nodes('RingBufferTarget/event[@name="xml_deadlock_report"]') AS X(XEvent)
ORDER BY DeadlockTime DESC;

**Why:** two sessions lock rows in opposite order; one is chosen as victim (Msg 1205). Run the A/B ordering from `lesson.ipynb` first, then read the graph. The XML shows both processes, the resources each held and wanted, and which was the victim.

## Exercise 5

Show the top 5 non-idle cumulative wait types. In a comment, name what each of your top 2 waits generally indicates.

In [ ]:
SELECT TOP 5 wait_type, wait_time_ms, waiting_tasks_count
FROM sys.dm_os_wait_stats
WHERE wait_type NOT IN ('SLEEP_TASK','LAZYWRITER_SLEEP','WAITFOR','XE_TIMER_EVENT',
        'REQUEST_FOR_DEADLOCK_SEARCH','CHECKPOINT_QUEUE','DIRTY_PAGE_POLL','BROKER_TASK_STOP',
        'SOS_WORK_DISPATCHER','LOGMGR_QUEUE','QDS_PERSIST_TASK_MAIN_LOOP_SLEEP',
        'DISPATCHER_QUEUE_SEMAPHORE','SP_SERVER_DIAGNOSTICS_SLEEP','BROKER_TO_FLUSH',
        'PWAIT_EXTENSIBILITY_CLEANUP_TASK','QDS_ASYNC_QUEUE')
ORDER BY wait_time_ms DESC;

**Why:** aggregate `sys.dm_os_wait_stats`, filtering idle waits. Common interpretations:
- `PAGEIOLATCH_*` -> waiting on data pages from disk (I/O or memory pressure).
- `LCK_M_*` -> blocking/locking contention.
- `CXPACKET`/`CXCONSUMER` -> parallelism coordination (often benign; investigate if dominant).
- `WRITELOG` -> transaction log write latency.